In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("Imports successful!")

In [ ]:
incoming_path = (
    "/Volumes/workspace/incremental_project/"
    "source_files/incoming"
)

batch_002 = """order_id,customer_id,order_status,order_amount,updated_at
ORD006,C106,processing,300.00,2026-10-02 09:00:00
ORD007,C107,shipped,200.00,2026-10-02 09:05:00
ORD008,C108,processing,400.00,2026-10-02 09:10:00
ORD002,C102,delivered,150.00,2026-10-02 09:15:00
ORD003,C103,shipped,450.00,2026-10-02 09:20:00
"""

batch_002_path = f"{incoming_path}/orders_batch_002.csv"

# Prevent accidentally replacing a file already ingested
existing_files = [f.name for f in dbutils.fs.ls(incoming_path)]

assert "orders_batch_002.csv" not in existing_files, (
    "Batch 002 already exists. Do not overwrite it."
)

dbutils.fs.put(batch_002_path, batch_002, overwrite=False)

print("Created:", batch_002_path)

In [ ]:
display(dbutils.fs.ls(incoming_path))

In [ ]:
# Project configuration
catalog = "workspace"
schema = "incremental_project"
volume = "source_files"

# Base storage location
base_path = f"/Volumes/{catalog}/{schema}/{volume}"

# Original CSV location
incoming_path = f"{base_path}/incoming"

# Corrected CSV location
clean_path = f"{base_path}/incoming_clean"

table_name = f"{catalog}.{schema}.orders_bronze_v3"
checkpoint_path = f"{base_path}/_checkpoints/orders_v3"
schema_path = f"{base_path}/_schemas/orders_v3"

print("Original source:", incoming_path)
print("Clean source:", clean_path)

Data contracts: A data contract defines the expected structure of incoming data. Here, we're enforcing five specific column names. Implemented at expected_columns = [...]

Data preparation parses incoming CSV records and implemented at csv.reader(...)

Schema normalization: We remove unwanted whitespace from column names so different files follow the same naming convention. Implemented at col.strip()

Data validation rejects an unexpected header. Implemented at assert cleaned_header == expected_columns

Record count validation requires 1 header + 5 data records and implmented at assert len(rows) == 6

Non destructive processing writes corrected copies separately. Implemented at destination = f" {clean_path}/{filename}"

Overwrite prevention avoids replacing an existing clean file and implemented at if filename not in existing_files

In [ ]:
import csv
import io

base_path = "/Volumes/workspace/incremental_project/source_files"

clean_path = f"{base_path}/incoming_clean"
dbutils.fs.mkdirs(clean_path)

expected_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_amount",
    "updated_at"
]

for batch in ["001", "002"]:
    filename = f"orders_batch_{batch}.csv"
    source_file = f"{incoming_path}/{filename}"
    destination = f"{clean_path}/{filename}"

    content = dbutils.fs.head(source_file, 100000)
    rows = list(csv.reader(io.StringIO(content)))

    assert len(rows) == 6, f"Unexpected row count in {filename}"

    rows[0] = [column.strip() for column in rows[0]]

    assert rows[0] == expected_columns, (
        f"Invalid CSV header: {rows[0]}"
    )

    output = io.StringIO()
    writer = csv.writer(output, lineterminator="\n")
    writer.writerows(rows)

    existing = [f.name for f in dbutils.fs.ls(clean_path)]

    if filename not in existing:
        dbutils.fs.put(
            destination,
            output.getvalue(),
            overwrite=False
        )

    print(f"Checked: {filename}")

Schema definition describes the overall data structure and exact code is StructType([...])

Column definition defines an individual field and exact code is StructField(...)

Data type treats incoming values as strings and exact code is StringType()

Nullability permits null values in that field and exact code is True

Schema inspection displays the schema in a compact form and exact code is .simpleString()

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

order_schema = StructType ([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_amount", StringType(), True),
    StructField("updated_at", StringType(), True)
])

Structured streaming constructs a streaming read. It enables processing across successive data batches rather than performing a one time read with spark.read. spark.readStream is how it is implemented

.format("cloudFiles") selects Auto Loader as streaming source. It can discover newly arriving files and track their ingestion state

.option("cloudFiles.format","csv") tells Auto Loader that source files are CSVs, not JSON or Parquet

.option("cloudFiles.schemaLocation", schema_path) configures the schema metadata location. This is distinct from the streaming checkpoint, which tracks execution progress

.option("cloudFiles.schemaEvolutionMode", "rescue")
.option("rescuedDataColumn", "_rescued_data") is unexpected new fields are captured as rescued data rather than automatically added as columns to our declared schema. We should monitor that column because rescued values indicate something requires investication

.schema(order_schema) is where the StructType object actually gets applied to the source

.load(clean_path) points auto loader at normalized CSV folder, not the original folder that previously caused inconsistent headers

readStream and cloudFiles are not the same thing. readStream is the Spark Structured Streaming reader. cloudFiles is Databricks' Auto Loader source plugged into that streaming framework

In [ ]:
orders_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn","_rescued_data")
    .option("header", "true")
    .schema(order_schema)
    .load(clean_path)
)

orders_stream.printSchema()

.withColumn("_ingested_at", F.current_timestamp()) adds the timestamp when Spark processes the data

.withColumn("_source_file", F.col("_metadata.file_path")) adds the physical path of original CSV file to every output record

withColumn means it produces a new DataFrame with an added or replaced column. It does not update source CSV file

updated_at is when the source system says the order changed and created where original CSV is

_ingested_at is when Spark processed the record

In [ ]:
bronze_stream = (
    orders_stream
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

bronze_stream.printSchema()

Streaming execution configures a streaming write and implemented at writeStream

Delta lake stores incoming data as a transactional Delta table and implemented at .format("delta")

Append only bronze preserves received records and implemented at .outputMode("append")

Checkpointing tracks stream progress and supports recovery and implemented at .option("checkpointLocation", checkpoint_path)

Incremental scheduled run processes currently available files and stops and implemented at .trigger(availableNow=True)

Table destination starts writing into the target table and implemented at .toTable(table_name)

Completion synchronization waits for the streaming query to finish and implemented at query.awaitTermination()

In [ ]:
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(table_name)
)

query.awaitTermination()

print("Bronze ingestion completed!")

spark.table(table_name) read an existing Delta table
.count() records count validation
.select("order_id").distinct() is business-key uniqueness analysis
.select("_source_file").distinct() is source file coverage and lineage validation
.filter(...isNotNull()) is data quality monitoring for rescued fields

In [ ]:
bronze_df = spark.table(table_name)

total_rows = bronze_df.count()

unique_orders = (
    bronze_df.select("order_id")
    .distinct()
    .count()
)

source_files = (
    bronze_df.select("_source_file")
    .distinct()
    .count()
)

rescued_rows = (
    bronze_df
    .filter(F.col("_rescued_data").isNotNull())
    .count()
)

print("Total rows:", total_rows)
print("Unique orders:", unique_orders)
print("Source files:", source_files)
print("Rows with rescued data:", rescued_rows)

In [ ]:
assert total_rows == 10
assert unique_orders == 8
assert source_files == 2
assert rescued_rows == 0

print("Step 3 validation passed!")

Restart safe incremental ingestion

Combination of Auto loader, checkpointLocation, and transactonal Delta streaming sink

Proves that with unchanged files and same stream state, rerunning ingestion does not add same 10 rows again

It does not prove that all future incoming files will contain unique orders

In [ ]:
rows_after_rerun = spark.table(table_name).count()

assert rows_after_rerun == 10

print("Checkpoint validation passed!")

Auto loader ingests newly arriving files incrementally rather than repeatedly processing unchanged files

10 rows and 8 unique IDs because bronze appends all source records, including updated versions of existing orders

Schema location describes source structure; checkpoint stores processing progress and recovery state